<a href="https://colab.research.google.com/github/tivanello/fase5/blob/main/notebooks/fase5_pergunta9_modelo_preditivo_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# BLOCO 1 — CLONAR O REPOSITÓRIO E IMPORTAR O DATAFRAME
# Objetivo:
# - Clonar o repositório no Colab
# - Localizar o arquivo tratado
# - Carregar o df_fase5 em um DataFrame
# ============================================================

import os
import pandas as pd

REPO_DIR = "/content/fase5"
ARQUIVO = f"{REPO_DIR}/data/processed/df_fase5.csv"

# Clonar o repositório apenas se ainda não existir
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/tivanello/fase5.git {REPO_DIR}
else:
    print("Repositório já existe no ambiente.")

# Carregar o arquivo
df_fase5 = pd.read_csv(ARQUIVO)

# Conferência inicial
print("Dimensão do dataframe:", df_fase5.shape)
display(df_fase5.head())

In [ ]:
# ============================================================
# BLOCO 2 — INSPEÇÃO DO ALVO COM BASE EM DEFASAGEM_FINAL
# Objetivo:
# - Confirmar a distribuição da coluna Defasagem_final
# - Criar a variável-alvo binária para a Pergunta 9
# ============================================================

print("Distribuição de Defasagem_final:")
print(df_fase5["Defasagem_final"].value_counts(dropna=False).sort_index())

df_modelo = df_fase5.copy()

# Alvo binário:
# 1 = aluno com defasagem
# 0 = aluno sem defasagem
df_modelo["alvo_risco_defasagem"] = (
    df_modelo["Defasagem_final"] > 0
).astype(int)

print("\nDistribuição da variável-alvo:")
print(df_modelo["alvo_risco_defasagem"].value_counts(dropna=False))

print("\nProporção da variável-alvo:")
print(df_modelo["alvo_risco_defasagem"].value_counts(normalize=True).round(4))

In [ ]:
df_modelo.info()


## Definição da variável-alvo

Para a modelagem da Pergunta 9, a variável-alvo foi construída a partir da coluna Defasagem_final (alvo_risco_defasagem).

### Adotou-se a seguinte regra de classificação:

* 1 = aluno com risco de defasagem (Defasagem_final > 0)

* 0 = aluno sem risco de defasagem (Defasagem_final <= 0)

Essa escolha permite representar de forma objetiva os casos em que o aluno se encontra abaixo da fase ideal esperada.

Como verificado a distribuição da variável-alvo mostrou forte desbalanceamento entre as classes, com aproximadamente 93,7% dos registros na classe sem risco (1) e 6,3% na classe com risco de defasagem (0). Em razão disso, a avaliação dos modelos deverá priorizar métricas mais adequadas para bases desbalanceadas, como precision, recall, F1-score e ROC-AUC, em vez de considerar apenas a acurácia.

Por fim, como Defasagem_final foi utilizada para definir o alvo, essa variável será excluída do conjunto de preditores para evitar vazamento de informação no treinamento do modelo.

In [ ]:
# ============================================================
# BLOCO 3 — DIAGNÓSTICO DAS COLUNAS PARA DEFINIÇÃO DO X
# Objetivo:
# - Recriar a variável-alvo
# - Analisar cada coluna quanto a tipo, nulos e cardinalidade
# - Sugerir quais colunas devem ficar ou sair do modelo
# ============================================================

import pandas as pd
import numpy as np

df_modelo = df_fase5.copy()

# Recriar alvo
df_modelo["alvo_risco_defasagem"] = (df_modelo["Defasagem_final"] > 0).astype(int)

# Colunas que já devem sair por regra de negócio / vazamento
colunas_excluir_forcadas = [
    "alvo_risco_defasagem",
    "Defasagem_final",
    "RA",
    "ra_num",
    "FaseIdeal_txt",
    "FaseIdeal_num",
    "Fase_num",
    "Fase_sufixo"
]

colunas_excluir_forcadas = [c for c in colunas_excluir_forcadas if c in df_modelo.columns]

# Montar diagnóstico
diagnostico = pd.DataFrame({
    "coluna": df_modelo.columns,
    "dtype": df_modelo.dtypes.astype(str).values,
    "nulos": df_modelo.isna().sum().values,
    "pct_nulos": (df_modelo.isna().mean() * 100).round(2).values,
    "qtd_unicos": df_modelo.nunique(dropna=True).values
})

# Tipo simplificado
diagnostico["tipo_variavel"] = np.where(
    diagnostico["dtype"].isin(["int64", "float64", "Int64", "Float64"]),
    "numerica",
    "categorica"
)

# Regras de decisão
def classificar_coluna(linha):
    col = linha["coluna"]
    pct_nulos = linha["pct_nulos"]
    qtd_unicos = linha["qtd_unicos"]
    tipo = linha["tipo_variavel"]

    if col in colunas_excluir_forcadas:
        return "EXCLUIR - vazamento/regra"

    if pct_nulos >= 85:
        return "EXCLUIR - nulos muito altos"

    if tipo == "categorica" and qtd_unicos >= 200:
        return "REVISAR - cardinalidade muito alta"

    if tipo == "categorica" and qtd_unicos == 1:
        return "EXCLUIR - sem variação"

    if tipo == "numerica" and qtd_unicos <= 1:
        return "EXCLUIR - sem variação"

    if pct_nulos >= 60:
        return "REVISAR - muitos nulos"

    return "MANTER - candidata"

diagnostico["decisao_inicial"] = diagnostico.apply(classificar_coluna, axis=1)

# Ordenar para facilitar leitura
diagnostico = diagnostico.sort_values(
    by=["decisao_inicial", "pct_nulos", "qtd_unicos"],
    ascending=[True, False, False]
).reset_index(drop=True)

print("Resumo da decisão inicial:")
print(diagnostico["decisao_inicial"].value_counts())

display(diagnostico)

# Listas auxiliares
colunas_manter = diagnostico.loc[
    diagnostico["decisao_inicial"] == "MANTER - candidata", "coluna"
].tolist()

colunas_revisar = diagnostico.loc[
    diagnostico["decisao_inicial"].str.contains("REVISAR", na=False), "coluna"
].tolist()

colunas_excluir = diagnostico.loc[
    diagnostico["decisao_inicial"].str.contains("EXCLUIR", na=False), "coluna"
].tolist()

print("\nColunas candidatas a permanecer:", len(colunas_manter))
print(colunas_manter)

print("\nColunas para revisão manual:", len(colunas_revisar))
print(colunas_revisar)

print("\nColunas sugeridas para exclusão:", len(colunas_excluir))
print(colunas_excluir)

## *Definição final do conjunto inicial de preditores*

A análise diagnóstica das colunas mostrou que a classificação automática entre “manter”, “revisar” e “excluir” é um bom ponto de partida, mas não é suficiente, por si só, para definir o conjunto final de variáveis do modelo.

Por esse motivo, foi realizada uma revisão analítica complementar, considerando não apenas critérios estatísticos, mas também o significado das variáveis no contexto do problema.

Com isso, optou-se por:

1) manter variáveis de perfil, contexto escolar, indicadores educacionais e medidas de desempenho;

2) excluir variáveis administrativas, identificadores de avaliadores e colunas consolidadas finais que poderiam introduzir ruído ou vazamento de informação;

3) deixar de fora, neste primeiro modelo, colunas com excesso de valores ausentes e baixo ganho interpretativo.

Essa decisão busca construir uma primeira versão do modelo com maior robustez, melhor interpretabilidade e menor risco de contaminação por variáveis excessivamente próximas do desfecho.

In [ ]:
# ============================================================
# BLOCO 3.1 — DEFINIÇÃO FINAL DO CONJUNTO X
# Objetivo:
# - Selecionar um conjunto inicial de preditores mais limpo
# - Remover colunas administrativas, consolidadas finais e ruído
# - Preparar X e y para treino e teste
# ============================================================

df_modelo = df_fase5.copy()

# Recriar alvo
df_modelo["alvo_risco_defasagem"] = (df_modelo["Defasagem_final"] > 0).astype(int)

# Conjunto inicial de variáveis escolhidas manualmente
colunas_x = [
    "Fase",
    "Turma",
    "Gênero",
    "Ano ingresso",
    "Instituição de ensino",
    "Nº Av",
    "IAA",
    "IEG",
    "IPS",
    "IDA",
    "IPV",
    "IAN",
    "ano_base",
    "INDE 22",
    "Pedra 22",
    "IPP",
    "Mat",
    "Por",
    "Idade"
]

# Manter apenas as colunas existentes
colunas_x = [col for col in colunas_x if col in df_modelo.columns]

X = df_modelo[colunas_x].copy()
y = df_modelo["alvo_risco_defasagem"].copy()

print("Dimensão de X:", X.shape)
print("Dimensão de y:", y.shape)

print("\nColunas finais selecionadas para o modelo:")
print(colunas_x)

print("\nTipos das colunas:")
display(X.dtypes.to_frame("tipo"))

print("\nPercentual de nulos nas colunas selecionadas:")
display(
    (X.isna().mean() * 100)
    .round(2)
    .sort_values(ascending=False)
    .to_frame("pct_nulos")
)

In [ ]:
X.info()


In [ ]:
X.head(100)

In [ ]:
# ============================================================
# BLOCO 3.2 — SEPARAÇÃO DAS COLUNAS NUMÉRICAS E CATEGÓRICAS
# Objetivo:
# - Identificar os grupos de variáveis para o pipeline
# ============================================================

colunas_numericas = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
colunas_categoricas = X.select_dtypes(include=["object"]).columns.tolist()

print("Colunas numéricas:")
print(colunas_numericas)

print("\nColunas categóricas:")
print(colunas_categoricas)

## *Verificação dos tipos das variáveis selecionadas*

Após a definição do conjunto inicial de preditores, foi realizada uma conferência visual dos tipos das colunas. O resultado mostrou uma composição coerente entre variáveis numéricas e categóricas, sem necessidade de conversões obrigatórias neste momento.

As colunas numéricas já se encontram em formato adequado para uso em pipeline de modelagem, enquanto as colunas categóricas poderão ser tratadas posteriormente com técnicas de codificação, como One-Hot Encoding.

As variáveis Ano ingresso e ano_base, embora representem informação temporal/discreta, serão mantidas inicialmente como numéricas, por simplicidade e por não comprometerem a construção do primeiro modelo.

Assim, nesta etapa, optou-se por não realizar transformações adicionais de tipo, concentrando o próximo passo na separação entre treino e teste e no tratamento automatizado dentro do pipeline.

In [ ]:
# ============================================================
# BLOCO 4 — SEPARAÇÃO ENTRE TREINO E TESTE
# Objetivo:
# - Dividir a base em treino e teste
# - Preservar a proporção da classe minoritária
# ============================================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:")
print(y_train.value_counts(normalize=True).round(4))

print("\ny_test:")
print(y_test.value_counts(normalize=True).round(4))

## *Separação entre treino e teste*

Nesta etapa, a base foi dividida em dois subconjuntos:

1) treino (X): utilizado para ajustar o modelo;

2) teste (y): utilizado para avaliar o desempenho em dados não vistos.

A divisão foi feita com estratificação da variável-alvo, o que garante a manutenção da proporção entre alunos com e sem risco de defasagem nos dois conjuntos.

O resultado mostrou que a distribuição da classe minoritária permaneceu praticamente igual em treino e teste:

* Treino: 6,31% de alunos com risco

* Teste: 6,27% de alunos com risco

Essa etapa é importante porque evita distorções na avaliação do modelo, especialmente em um problema com classes desbalanceadas como este.

In [ ]:
# ============================================================
# BLOCO 5 — PIPELINE DE PRÉ-PROCESSAMENTO + MODELO BASELINE
# Objetivo:
# - Tratar nulos
# - Codificar variáveis categóricas
# - Treinar um primeiro modelo baseline
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

# Separar colunas numéricas e categóricas
colunas_numericas = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
colunas_categoricas = X_train.select_dtypes(include=["object"]).columns.tolist()

print("Colunas numéricas:", colunas_numericas)
print("\nColunas categóricas:", colunas_categoricas)

# Pipeline para variáveis numéricas
transformador_numerico = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# Pipeline para variáveis categóricas
transformador_categorico = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Pré-processador geral
preprocessador = ColumnTransformer(transformers=[
    ("num", transformador_numerico, colunas_numericas),
    ("cat", transformador_categorico, colunas_categoricas)
])

# Modelo baseline
modelo_rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    class_weight="balanced"
)

# Pipeline final
pipeline_rf = Pipeline(steps=[
    ("preprocessador", preprocessador),
    ("modelo", modelo_rf)
])

# Treinamento
pipeline_rf.fit(X_train, y_train)

print("Modelo treinado com sucesso.")

## *Estrutura do pipeline de modelagem*

A saída do bloco confirmou a separação das variáveis em dois grupos:

    a) numéricas: indicadores, notas, idade e variáveis de contexto temporal;

    b) categóricas: fase, turma, gênero, instituição de ensino e classificação da pedra.

Com isso, o pipeline conseguiu aplicar o pré-processamento adequado para cada tipo de dado antes do treinamento.

O modelo Random Forest foi então ajustado com sucesso sobre a base de treino, já incorporando:

* tratamento de valores ausentes;

* codificação das variáveis categóricas;

* balanceamento das classes por meio de class_weight="balanced".

Essa etapa conclui a preparação do primeiro modelo baseline, deixando a base pronta para a avaliação de desempenho no conjunto de teste.

In [ ]:
# ============================================================
# BLOCO 6 — AVALIAÇÃO DO MODELO BASELINE
# Objetivo:
# - Gerar previsões no conjunto de teste
# - Avaliar acurácia, matriz de confusão e relatório de classificação
# - Calcular ROC-AUC e plotar a curva ROC
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    RocCurveDisplay
)
import matplotlib.pyplot as plt

# Previsões
y_pred = pipeline_rf.predict(X_test)
y_prob = pipeline_rf.predict_proba(X_test)[:, 1]

# Métricas
acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print("Acurácia:", round(acc, 4))
print("ROC-AUC :", round(auc, 4))

print("\nMatriz de confusão:")
print(confusion_matrix(y_test, y_pred))

print("\nRelatório de classificação:")
print(classification_report(y_test, y_pred, digits=4))

# Curva ROC
RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title("Curva ROC - Random Forest")
plt.show()

## **Conclusão da avaliação inicial**

O modelo Random Forest apresentou desempenho global muito bom, com:

* Acurácia: 95,71%

* ROC-AUC: 95,33%

Esses números mostram que a modelagem inicial foi bem-sucedida. No entanto, como o problema é desbalanceado e o foco prático está na identificação de alunos em risco de defasagem, a métrica mais sensível neste contexto é **o recall da classe 1, que ficou em 44,74%**.

Assim, apesar do bom desempenho geral, ainda há espaço para melhoria. Os próximos passos devem buscar aumentar a capacidade de detecção da classe de risco, mesmo que isso implique aceitar um pequeno aumento de falsos positivos.

## **IMPORTANTE**

**Por que o recall da classe 1 é tão importante neste problema**

Neste projeto, a classe 1 representa os alunos com risco de defasagem, ou seja, justamente o grupo que a análise preditiva deseja identificar com antecedência para possibilitar ações de acompanhamento e intervenção.

Por esse motivo, o recall da classe 1 é uma métrica especialmente relevante. Ele mede a proporção de alunos realmente em risco que o modelo conseguiu identificar corretamente.

A fórmula, em termos simples, é:

* recall = verdadeiros positivos / (verdadeiros positivos + falsos negativos)

No resultado obtido:

* 17 alunos em risco foram corretamente identificados;

* 21 alunos em risco não foram detectados.

Assim, o **recall da classe 1 ficou em aproximadamente 44,74%, o que significa que o modelo encontrou menos da metade dos casos reais de risco presentes no conjunto de teste**.

Essa métrica é crítica porque, neste contexto, um falso negativo é mais preocupante do que um falso positivo:

* Falso negativo: o aluno está em risco, mas o modelo não sinaliza;

* Falso positivo: o aluno é sinalizado como risco, mas na prática não estava.

Em aplicações educacionais preventivas, deixar de identificar um aluno realmente em risco pode significar perder a oportunidade de agir a tempo. Já um falso positivo, embora gere atenção desnecessária em alguns casos, costuma ser menos grave do que não detectar quem realmente precisa de apoio.

Por isso, mesmo com acurácia elevada e ROC-AUC excelente, a análise do modelo precisa dar atenção especial ao recall da classe 1. Em outras palavras: não basta o modelo ser bom no geral; ele precisa ser bom, principalmente, em encontrar os alunos que mais importam para a intervenção.

In [ ]:
# ============================================================
# BLOCO 7 — AVALIAÇÃO DE DIFERENTES LIMIARES DE DECISÃO
# Objetivo:
# - Testar diferentes thresholds para a classe 1
# - Comparar precision, recall e f1-score
# - Apoiar a escolha de um limiar mais adequado
# ============================================================

import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

limiares = [0.50, 0.45, 0.40, 0.35, 0.30, 0.25, 0.20]

resultados_limiares = []

for limiar in limiares:
    y_pred_limiar = (y_prob >= limiar).astype(int)

    precision = precision_score(y_test, y_pred_limiar, zero_division=0)
    recall = recall_score(y_test, y_pred_limiar, zero_division=0)
    f1 = f1_score(y_test, y_pred_limiar, zero_division=0)
    matriz = confusion_matrix(y_test, y_pred_limiar)

    resultados_limiares.append({
        "limiar": limiar,
        "precision_classe_1": round(precision, 4),
        "recall_classe_1": round(recall, 4),
        "f1_classe_1": round(f1, 4),
        "tn": matriz[0, 0],
        "fp": matriz[0, 1],
        "fn": matriz[1, 0],
        "tp": matriz[1, 1]
    })

df_limiares = pd.DataFrame(resultados_limiares)
display(df_limiares)

## **Escolha do limiar com foco preventivo**

A análise dos limiares mostrou que o **valor de 0,20** produziu o maior recall da classe 1, **alcançando 71,05% dos alunos em risco no conjunto de teste**.

Embora esse limiar apresente redução de precisão em relação a pontos de corte mais altos, sua adoção é justificável neste contexto, pois **o objetivo principal do modelo é ampliar a identificação precoce de estudantes com risco de defasagem**, aumentando o Falso positivo de alunos que não estão em RISCO e cairem nessa situação.

No contexto educacional, um **falso negativo** tende a ser mais preocupante do que um falso positivo, pois representa um aluno realmente em risco que deixa de ser sinalizado para acompanhamento. Já o falso positivo, embora gere atenção adicional, pode ser absorvido com menor impacto em uma estratégia preventiva.

Dessa forma, optou-se por adotar o **limiar 0,20** como ponto de corte final do modelo, privilegiando a sensibilidade na detecção dos casos de risco.

In [ ]:
# ============================================================
# BLOCO 8 — AVALIAÇÃO FINAL COM LIMIAR AJUSTADO
# Objetivo:
# - Aplicar o limiar final de 0,20
# - Recalcular matriz de confusão e métricas
# - Registrar o desempenho final do modelo
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

# Limiar final adotado
limiar_final = 0.20

# Previsões ajustadas
y_pred_final = (y_prob >= limiar_final).astype(int)

# Métricas finais
acc_final = accuracy_score(y_test, y_pred_final)
matriz_final = confusion_matrix(y_test, y_pred_final)
relatorio_final = classification_report(y_test, y_pred_final, digits=4)

print("Limiar final adotado:", limiar_final)
print("Acurácia final:", round(acc_final, 4))

print("\nMatriz de confusão final:")
print(matriz_final)

print("\nRelatório de classificação final:")
print(relatorio_final)

## **Análise da Matriz**

Com a adoção do limiar 0,20, a matriz de confusão final apresentou os seguintes resultados:

* 545 verdadeiros negativos: alunos sem risco corretamente classificados;

* 27 verdadeiros positivos: alunos em risco corretamente identificados;

* 23 falsos positivos: alunos sem risco classificados como risco;

* 11 falsos negativos: alunos em risco que não foram identificados pelo modelo.

Comparando com o limiar padrão de 0,50, houve uma melhora importante na capacidade de detectar alunos em risco:

* os verdadeiros positivos subiram de 17 para 27;

* os falsos negativos caíram de 21 para 11.

Isso mostra que o ajuste do limiar tornou o modelo mais sensível à classe de risco, o que é desejável em um cenário de prevenção.

## **Interpretação do Relatório de Classificação**

**Classe 0 — sem risco de defasagem**

* Precision: 0,9802

* Recall: 0,9595

* F1-score: 0,9698

O modelo continuou com desempenho muito forte para a classe majoritária, mantendo alta capacidade de classificar corretamente os alunos sem risco.

**Classe 1 — com risco de defasagem**

* Precision: 0,5400

* Recall: 0,7105

* F1-score: 0,6136

Para a classe de risco, o comportamento ficou mais alinhado ao objetivo do projeto. O modelo passou a identificar 71,05% dos casos reais de risco, o que representa um avanço importante em relação ao limiar padrão.

Houve queda de precisão, o que significa aumento de falsos positivos, mas esse efeito é aceitável dentro de uma estratégia preventiva, em que vale mais sinalizar alguns casos extras do que deixar passar alunos realmente vulneráveis.

## **Comparação entre o limiar padrão e o limiar ajustado**

Comparando os dois cenários:

**Limiar 0,50**

* Precision classe 1: 0,7727

* Recall classe 1: 0,4474

* F1-score classe 1: 0,5667

**Limiar 0,20**

* Precision classe 1: 0,5400

* Recall classe 1: 0,7105

* F1-score classe 1: 0,6136

Essa comparação mostra que o limiar 0,20 sacrificou parte da precisão, mas entregou ganho relevante em recall e também melhorou o F1-score, tornando o modelo mais útil para identificação precoce de risco.

## **Conclusão final do modelo**

O modelo final, baseado em Random Forest com limiar ajustado para 0,20, apresentou desempenho global sólido e, principalmente, maior sensibilidade para identificar alunos em risco de defasagem.

Os principais resultados finais foram:

* Acurácia: 94,39%

* Recall da classe 1: 71,05%

* F1-score da classe 1: 61,36%

Esses resultados indicam que o modelo é capaz de funcionar como ferramenta de apoio para sinalização preventiva de alunos com maior probabilidade de defasagem, oferecendo uma base objetiva para priorização de acompanhamento pedagógico.

Em termos práticos, o ajuste do limiar tornou o modelo mais aderente ao problema de negócio: em vez de apenas acertar muito no geral, ele passou a cumprir melhor o papel de encontrar quem realmente precisa de atenção.

In [ ]:
# ============================================================
# BLOCO 9 — IMPORTÂNCIA DAS VARIÁVEIS
# Objetivo:
# - Recuperar os nomes das variáveis após o pré-processamento
# - Extrair a importância calculada pelo Random Forest
# - Exibir ranking das variáveis mais relevantes
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt

# Recuperar o pré-processador e o modelo treinado
preprocessador_ajustado = pipeline_rf.named_steps["preprocessador"]
modelo_ajustado = pipeline_rf.named_steps["modelo"]

# Nomes finais das variáveis após transformação
nomes_features = preprocessador_ajustado.get_feature_names_out()

# Importâncias do modelo
importancias = modelo_ajustado.feature_importances_

# DataFrame de importância
df_importancias = pd.DataFrame({
    "variavel": nomes_features,
    "importancia": importancias
}).sort_values("importancia", ascending=False).reset_index(drop=True)

# Exibir top 20
print("Top 20 variáveis mais importantes:")
display(df_importancias.head(20))

# Gráfico
top_n = 15
df_top = df_importancias.head(top_n).sort_values("importancia", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(df_top["variavel"], df_top["importancia"])
plt.title("Top 15 variáveis mais importantes - Random Forest")
plt.xlabel("Importância")
plt.ylabel("Variável")
plt.tight_layout()
plt.show()

Para fins de interpretação analítica e registro metodológico, os indicadores considerados no projeto possuem o seguinte significado:

* IAN — indicador relacionado à defasagem de aprendizagem / nível de atenção acadêmica
* IDA — indicador de desempenho acadêmico (aprendizagem)
* IEG — indicador de engajamento do aluno
* IAA — indicador de autoavaliação do aluno
* IPV — indicador associado ao ponto de virada / proximidade de mudança de trajetória
* IPS — indicador de percepção ou componente socioemocional, conforme definição da base
* IPP — indicador ligado ao protagonismo / permanência / participação, conforme o conceito adotado
* INDE — índice consolidado de desenvolvimento educacional

Essas definições serão utilizadas como referência para a interpretação das variáveis no modelo preditivo e na análise dos fatores associados ao risco de defasagem.

Com issoa a análise de importância das variáveis mostrou que o modelo concentrou maior peso em indicadores educacionais e contextuais diretamente relacionados à trajetória do aluno.

O destaque principal foi o IAN, indicador associado à defasagem de aprendizagem e ao nível de atenção acadêmica, sugerindo forte relação com o risco previsto pelo modelo. Em seguida, apareceram variáveis como Idade, INDE 22, Nº Av, IPV, IEG, IDA, IAA, IPS e IPP, reforçando que a previsão do risco não depende de um único fator, mas de uma combinação entre desempenho, engajamento, autoavaliação, participação e histórico educacional.

Também surgiram variáveis ligadas à instituição de ensino, indicando que o contexto escolar pode influenciar a probabilidade de defasagem observada pelo modelo.

De forma geral, o resultado mostra que o modelo aprendeu com sinais coerentes com o problema analisado, apoiando-se principalmente em dimensões pedagógicas e de acompanhamento educacional.

________________________________________________________________________________
##  **Teste de robustez do modelo sem a variável IAN**

O objetivo dessa etapa é verificar se o desempenho do modelo se mantém em nível satisfatório mesmo sem o principal atributo. Esse teste é importante para avaliar a robustez da solução e reduzir o risco de dependência excessiva de uma única variável.

Se o modelo continuar apresentando bom desempenho sem o IAN, isso indica que a previsão também está sendo sustentada por outros sinais relevantes da base, como desempenho acadêmico, engajamento, autoavaliação, participação e contexto escolar.


In [ ]:
# ============================================================
# BLOCO 10 — MODELO SEM A VARIÁVEL IAN
# Objetivo:
# - Remover a variável IAN do conjunto de entrada
# - Treinar novamente o Random Forest
# - Avaliar o desempenho no conjunto de teste
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score

# Remover IAN do treino e do teste
X_train_sem_ian = X_train.drop(columns=["IAN"])
X_test_sem_ian = X_test.drop(columns=["IAN"])

# Separar colunas numéricas e categóricas
colunas_numericas_sem_ian = X_train_sem_ian.select_dtypes(include=["int64", "float64"]).columns.tolist()
colunas_categoricas_sem_ian = X_train_sem_ian.select_dtypes(include=["object"]).columns.tolist()

print("Colunas numéricas:", colunas_numericas_sem_ian)
print("\nColunas categóricas:", colunas_categoricas_sem_ian)

# Pipeline numérico
transformador_numerico_sem_ian = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# Pipeline categórico
transformador_categorico_sem_ian = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Pré-processador
preprocessador_sem_ian = ColumnTransformer(transformers=[
    ("num", transformador_numerico_sem_ian, colunas_numericas_sem_ian),
    ("cat", transformador_categorico_sem_ian, colunas_categoricas_sem_ian)
])

# Modelo
modelo_rf_sem_ian = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

# Pipeline final
pipeline_rf_sem_ian = Pipeline(steps=[
    ("preprocessador", preprocessador_sem_ian),
    ("modelo", modelo_rf_sem_ian)
])

# Treinamento
pipeline_rf_sem_ian.fit(X_train_sem_ian, y_train)

# Probabilidades e previsão com limiar 0,20
y_prob_sem_ian = pipeline_rf_sem_ian.predict_proba(X_test_sem_ian)[:, 1]
y_pred_sem_ian = (y_prob_sem_ian >= 0.20).astype(int)

# Métricas
acc_sem_ian = accuracy_score(y_test, y_pred_sem_ian)
auc_sem_ian = roc_auc_score(y_test, y_prob_sem_ian)

print("\nAcurácia:", round(acc_sem_ian, 4))
print("ROC-AUC :", round(auc_sem_ian, 4))

print("\nMatriz de confusão:")
print(confusion_matrix(y_test, y_pred_sem_ian))

print("\nRelatório de classificação:")
print(classification_report(y_test, y_pred_sem_ian, digits=4))

In [ ]:
# ============================================================
# BLOCO 10.1 — TESTE DE DIFERENTES LIMIARES NO MODELO SEM IAN
# Objetivo:
# - Avaliar diferentes thresholds no modelo sem IAN
# - Comparar precision, recall e f1-score
# - Escolher o melhor limiar para esse novo cenário
# ============================================================

import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

limiares_sem_ian = [0.50, 0.45, 0.40, 0.35, 0.30, 0.25, 0.20]

resultados_limiares_sem_ian = []

for limiar in limiares_sem_ian:
    y_pred_limiar_sem_ian = (y_prob_sem_ian >= limiar).astype(int)

    precision = precision_score(y_test, y_pred_limiar_sem_ian, zero_division=0)
    recall = recall_score(y_test, y_pred_limiar_sem_ian, zero_division=0)
    f1 = f1_score(y_test, y_pred_limiar_sem_ian, zero_division=0)
    matriz = confusion_matrix(y_test, y_pred_limiar_sem_ian)

    resultados_limiares_sem_ian.append({
        "limiar": limiar,
        "precision_classe_1": round(precision, 4),
        "recall_classe_1": round(recall, 4),
        "f1_classe_1": round(f1, 4),
        "tn": matriz[0, 0],
        "fp": matriz[0, 1],
        "fn": matriz[1, 0],
        "tp": matriz[1, 1]
    })

df_limiares_sem_ian = pd.DataFrame(resultados_limiares_sem_ian)
display(df_limiares_sem_ian)

## **Escolha do limiar no modelo sem IAN**

Após o teste de diferentes limiares no modelo sem a variável IAN, observou-se que o ponto de corte 0,25 apresentou o melhor equilíbrio entre precision, recall e F1-score para a classe de risco.

Os resultados nesse limiar foram:

* precision: 0,6154

* recall: 0,6316

* F1-score: 0,6234

Embora o limiar 0,20 tenha produzido recall ligeiramente maior, o valor de 0,25 apresentou melhor equilíbrio geral e maior F1-score, sendo mais adequado para representar o desempenho do modelo sem IAN.

Dessa forma, o limiar 0,25 foi adotado como ponto de corte final para a avaliação comparativa dessa segunda versão do modelo.

In [ ]:
# ============================================================
# BLOCO 10.2 — AVALIAÇÃO FINAL DO MODELO SEM IAN
# Objetivo:
# - Aplicar o limiar final de 0,25 no modelo sem IAN
# - Recalcular métricas e matriz de confusão
# - Registrar a versão final comparável do modelo sem IAN
# ============================================================

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Limiar final do modelo sem IAN
limiar_final_sem_ian = 0.25

# Previsões finais
y_pred_final_sem_ian = (y_prob_sem_ian >= limiar_final_sem_ian).astype(int)

# Métricas finais
acc_final_sem_ian = accuracy_score(y_test, y_pred_final_sem_ian)
matriz_final_sem_ian = confusion_matrix(y_test, y_pred_final_sem_ian)
relatorio_final_sem_ian = classification_report(y_test, y_pred_final_sem_ian, digits=4)

print("Limiar final adotado (sem IAN):", limiar_final_sem_ian)
print("Acurácia final:", round(acc_final_sem_ian, 4))

print("\nMatriz de confusão final:")
print(matriz_final_sem_ian)

print("\nRelatório de classificação final:")
print(relatorio_final_sem_ian)

## **Comparação entre os modelos com e sem IAN**

A comparação mostrou que os dois modelos apresentaram desempenho satisfatório, mas com características distintas.

O **modelo com IAN** apresentou maior capacidade de detecção da classe de risco, alcançando recall de 71,05%, o que o torna mais adequado quando a prioridade é ampliar a identificação de alunos potencialmente vulneráveis.

Por outro lado, o **modelo sem IAN** apresentou melhor equilíbrio geral, com acurácia de 95,21%, precision de 61,54% e F1-score de 62,34% para a classe 1. Esse resultado sugere uma solução mais robusta e menos dependente de uma única variável dominante.



## **Conclusão da Pergunta 9**

A modelagem preditiva desenvolvida para estimar o risco de defasagem dos alunos mostrou que é possível identificar, com bom nível de desempenho, padrões associados à vulnerabilidade educacional a partir dos indicadores disponíveis na base.

Os testes realizados com **Random Forest** indicaram que o problema pode ser tratado de forma eficaz mesmo em um cenário de forte desbalanceamento entre as classes, desde que a avaliação priorize métricas adequadas, especialmente recall, precision e F1-score para a classe de risco.

**Dois cenários foram comparados:**

1) modelo com IAN, mais sensível para detectar alunos em risco;

2) modelo sem IAN, mais equilibrado e metodologicamente mais robusto.

O modelo com IAN alcançou maior capacidade de detecção dos casos de risco, com recall de 71,05%, sendo mais indicado quando o objetivo principal é maximizar a identificação preventiva de alunos vulneráveis.

Já o modelo sem IAN apresentou o melhor equilíbrio geral, com:

* acurácia de 95,21%

* precision de 61,54%

* recall de 63,16%

* F1-score de 62,34%

Esse resultado mostra que a solução continua forte mesmo sem depender da variável mais dominante (IAN), o que reforça sua consistência analítica.

De forma geral, a análise permite concluir que o uso de modelagem preditiva pode apoiar a Passos Mágicos na identificação antecipada de alunos com maior probabilidade de defasagem, contribuindo para priorização de acompanhamento, intervenções pedagógicas e ações preventivas mais direcionadas.

**Como encaminhamento prático, o modelo sem IAN pode ser adotado como versão principal da solução por apresentar maior equilíbrio e robustez**, enquanto o modelo com IAN pode ser mantido como alternativa complementar em cenários nos quais a prioridade seja ampliar ao máximo a sensibilidade para detecção de risco.

--------------------------------------------------------------------------------
## **Salvando o modelo final para uso no Streamlit**

Após a definição do modelo final, será necessário salvar em arquivo os objetos que serão utilizados na aplicação.

Nesta etapa serão armazenados:

* o pipeline treinado, contendo o pré-processamento e o modelo;

* o limiar de decisão final adotado;

* a lista de colunas de entrada esperadas pelo modelo.

Essa abordagem facilita a reutilização da solução preditiva fora do notebook, garantindo que o Streamlit aplique exatamente o mesmo fluxo utilizado no treinamento.

In [ ]:
# ============================================================
# BLOCO 11 — SALVAR MODELO FINAL PARA USO NO STREAMLIT
# Objetivo:
# - Salvar o pipeline treinado
# - Salvar o limiar final
# - Salvar a lista de colunas esperadas
# ============================================================

import os
import joblib

# Pasta onde os artefatos serão salvos
PASTA_MODELO = "/content/fase5/models"
os.makedirs(PASTA_MODELO, exist_ok=True)

# Definir o artefato final
artefato_modelo = {
    "pipeline": pipeline_rf_sem_ian,
    "limiar": 0.25,
    "colunas_entrada": X_train_sem_ian.columns.tolist(),
    "nome_modelo": "RandomForest_sem_IAN",
    "versao": "v1"
}

# Caminho do arquivo
CAMINHO_MODELO = os.path.join(PASTA_MODELO, "modelo_risco_defasagem_rf_sem_ian.joblib")

# Salvar
joblib.dump(artefato_modelo, CAMINHO_MODELO)

print("Modelo salvo com sucesso em:")
print(CAMINHO_MODELO)

## **Versionando o modelo no GitHub**

Após salvar o artefato do modelo em arquivo .joblib, o próximo passo é incluí-lo no repositório GitHub do projeto.

Nesta etapa, o arquivo será:

1) verificado no diretório do repositório;

2) adicionado ao controle de versão;

3) registrado com commit;

4) enviado ao GitHub com push.

Isso permite reutilizar o modelo posteriormente no Streamlit e manter a solução preditiva versionada junto com o restante do projeto.

In [ ]:
# ============================================================
# BLOCO 11.1 — CONFIGURAÇÃO E CONFERÊNCIA
# Objetivo:
# - Definir caminhos
# - Configurar usuário do Git
# - Verificar se o arquivo do modelo existe
# ============================================================

import os
import subprocess
from google.colab import userdata

REPO_DIR = "/content/fase5"
ARQUIVO_MODELO = "models/modelo_risco_defasagem_rf_sem_ian.joblib"
TOKEN = userdata.get("ITHUB_TOKEN")   # use "GITHUB_TOKEN" se esse for o nome real do seu secret

REMOTE_URL = "https://github.com/tivanello/fase5.git"
REMOTE_AUTH = REMOTE_URL.replace("https://", f"https://x-access-token:{TOKEN}@")

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    print("CMD:", " ".join(cmd))
    if r.stdout.strip():
        print("\nSTDOUT:\n", r.stdout)
    if r.stderr.strip():
        print("\nSTDERR:\n", r.stderr)
    print("-" * 80)
    return r

# Configurar identidade do Git no repositório
run(["git", "-C", REPO_DIR, "config", "user.name", "tivanello"])
run(["git", "-C", REPO_DIR, "config", "user.email", "tivanello@gmail.com"])

# Conferir arquivo
caminho_completo = os.path.join(REPO_DIR, ARQUIVO_MODELO)
print("Arquivo existe?", os.path.exists(caminho_completo))
print("Caminho:", caminho_completo)

# Verificar branch atual
run(["git", "-C", REPO_DIR, "branch", "--show-current"])

# Verificar status
run(["git", "-C", REPO_DIR, "status", "--short"])

In [ ]:
# ============================================================
# BLOCO 11.2 — ADICIONAR E COMMITAR O MODELO
# Objetivo:
# - Adicionar o arquivo .joblib ao Git
# - Criar o commit local
# ============================================================

run(["git", "-C", REPO_DIR, "add", ARQUIVO_MODELO])
run(["git", "-C", REPO_DIR, "status", "--short"])
run([
    "git", "-C", REPO_DIR, "commit",
    "-m", "Adiciona modelo preditivo Random Forest sem IAN para uso no Streamlit"
])

In [ ]:
# ============================================================
# BLOCO 11.3 — SINCRONIZAR COM O REMOTO
# Objetivo:
# - Buscar atualizações do GitHub
# - Reaplicar o commit local por cima do remoto
# ============================================================

run(["git", "-C", REPO_DIR, "checkout", "main"])
run(["git", "-C", REPO_DIR, "fetch", REMOTE_AUTH, "main"])
run(["git", "-C", REPO_DIR, "pull", "--rebase", REMOTE_AUTH, "main"])

In [ ]:
# ============================================================
# BLOCO 11.4 — ENVIAR PARA O GITHUB
# Objetivo:
# - Fazer o push autenticado para a branch main
# ============================================================

run(["git", "-C", REPO_DIR, "push", REMOTE_AUTH, "main"])
run(["git", "-C", REPO_DIR, "status", "--short"])